# Chapter 12: Cross-Validation and Model Evaluation Techniques

**Book:** *scikit-learn Cookbook, Third Edition* – John Sukup  
**Chapter summary and practical examples**

This notebook summarizes the chapter's validation strategies, scoring methods, and model-selection workflow, with runnable examples.

## Learning Objectives

- Explain why a single train/test split may give an unstable estimate of performance.
- Compare K-fold, Stratified K-fold, leave-one-out, and nested cross-validation.
- Use `cross_val_score()`, `cross_validate()`, and `GridSearchCV()`.
- Choose evaluation metrics that fit the task and class distribution.
- Tune hyperparameters while keeping final evaluation data separate.
- Reason about whether a model is likely to generalize to new observations.

## Environment Setup

Cross-validation repeatedly trains and evaluates models on different data subsets. The examples below use reproducible synthetic classification data and scikit-learn's built-in diabetes regression dataset.

In [1]:
import numpy as np
from sklearn.datasets import make_classification, load_diabetes
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.model_selection import (
    KFold, StratifiedKFold, LeaveOneOut, cross_val_score,
    cross_validate, GridSearchCV, RandomizedSearchCV,
    train_test_split
)
from sklearn.metrics import accuracy_score, classification_report
from scipy.stats import loguniform

RANDOM_STATE = 2024


## 12.1 Introduction to Cross-Validation

Cross-validation estimates how a model may perform on data it has not trained on. In K-fold validation, the dataset is split into *k* parts; each part serves as the validation fold once, while the other folds are used for training. Averaging the fold scores usually gives a more informative estimate than relying on one arbitrary split.

For classification, **StratifiedKFold** attempts to preserve the class proportions in every fold. This is especially helpful when class frequencies are uneven.

In [2]:
X, y = make_classification(
    n_samples=500, n_features=10, n_informative=8,
    n_redundant=2, weights=[0.7, 0.3], random_state=RANDOM_STATE
)

kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

kfold_scores = cross_val_score(LogisticRegression(max_iter=1000), X, y, cv=kf, scoring="accuracy")
strat_scores = cross_val_score(LogisticRegression(max_iter=1000), X, y, cv=skf, scoring="accuracy")

print(f"KFold mean accuracy: {kfold_scores.mean():.3f} (std {kfold_scores.std():.3f})")
print(f"StratifiedKFold mean accuracy: {strat_scores.mean():.3f} (std {strat_scores.std():.3f})")


KFold mean accuracy: 0.754 (std 0.031)
StratifiedKFold mean accuracy: 0.768 (std 0.035)


## 12.2 Advanced Cross-Validation Methods

**Leave-One-Out Cross-Validation (LOOCV)** holds out one observation at a time and trains on all remaining observations. It makes extensive use of a small dataset but can require many model fits.

**Nested cross-validation** separates hyperparameter tuning from performance estimation. The inner loop selects settings; the outer loop estimates performance using data not used by that tuning step. This helps avoid an overly optimistic score caused by selecting settings on the same folds used to report performance.

In [3]:
diabetes = load_diabetes()
X_reg, y_reg = diabetes.data, diabetes.target

loo_scores = cross_val_score(
    Ridge(alpha=1.0), X_reg, y_reg,
    cv=LeaveOneOut(), scoring="neg_mean_squared_error"
)
print(f"LOOCV mean MSE: {-loo_scores.mean():.2f}")

inner_search = GridSearchCV(
    Ridge(), param_grid={"alpha": [0.01, 0.1, 1.0, 10.0, 100.0]},
    cv=5, scoring="neg_mean_squared_error"
)
nested_scores = cross_val_score(
    inner_search, X_reg, y_reg, cv=5, scoring="neg_mean_squared_error"
)
print(f"Nested CV mean MSE: {-nested_scores.mean():.2f}")


LOOCV mean MSE: 3327.66
Nested CV mean MSE: 3004.95


## 12.3 Implementing Cross-Validation in scikit-learn

- `cross_val_score()` returns a score for each fold using one scoring metric.
- `cross_validate()` can collect multiple metrics and timing information.
- `GridSearchCV()` evaluates the specified hyperparameter combinations with cross-validation and keeps the best-performing configuration according to the chosen scoring rule.

Metric choice matters. Accuracy can be misleading when one class dominates, so consider precision, recall, and F1 where the costs of false positives and false negatives differ.

In [4]:
cv_results = cross_validate(
    LogisticRegression(max_iter=1000), X, y, cv=StratifiedKFold(
        n_splits=5, shuffle=True, random_state=RANDOM_STATE
    ),
    scoring=["accuracy", "precision", "recall", "f1"],
    return_train_score=False
)

for metric in ["accuracy", "precision", "recall", "f1"]:
    values = cv_results[f"test_{metric}"]
    print(f"Mean {metric}: {values.mean():.3f} (std {values.std():.3f})")


Mean accuracy: 0.768 (std 0.035)
Mean precision: 0.690 (std 0.064)
Mean recall: 0.465 (std 0.151)
Mean f1: 0.541 (std 0.110)


## 12.4 Model Selection and Hyperparameter Tuning

Hyperparameters control how an estimator learns but are not usually learned directly from the training examples. Grid search evaluates every supplied combination; randomized search samples combinations from specified distributions and can be more practical for large search spaces.

Keep a final test set aside until model and hyperparameter choices are complete. Cross-validation scores support selection, while the untouched test set offers a final check of performance.

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=RANDOM_STATE
)

param_grid = {"C": [0.001, 0.01, 0.1, 1, 10, 100]}
grid = GridSearchCV(
    LogisticRegression(max_iter=1000), param_grid,
    cv=5, scoring="f1"
)
grid.fit(X_train, y_train)

print("Grid search best parameters:", grid.best_params_)
print(f"Best mean CV F1: {grid.best_score_:.3f}")
print(f"Held-out test F1: {grid.score(X_test, y_test):.3f}")


Grid search best parameters: {'C': 0.1}
Best mean CV F1: 0.479
Held-out test F1: 0.642


### Randomized Search

Randomized search is useful when a parameter can take many possible values or when evaluating every combination would be expensive. The number of trials is controlled by `n_iter`. The best search strategy depends on the size of the search space and the available compute budget.

In [6]:
random_search = RandomizedSearchCV(
    LogisticRegression(max_iter=1000),
    param_distributions={"C": loguniform(1e-3, 1e2)},
    n_iter=12, cv=5, scoring="f1", random_state=RANDOM_STATE
)
random_search.fit(X_train, y_train)
print("Randomized search best parameters:", random_search.best_params_)
print(f"Best mean CV F1: {random_search.best_score_:.3f}")


Randomized search best parameters: {'C': np.float64(0.17437295301054892)}
Best mean CV F1: 0.477


## 12.5 Evaluating Model Generalizability

Generalization is the ability to perform well on new observations from the intended deployment setting. A model that scores highly on training data but substantially worse on validation data may be overfitting. Low scores on both can indicate underfitting, weak features, noisy labels, or a task that is difficult to predict.

A credible evaluation should:

- Use splits that reflect how the model will be used (for example, grouped or time-aware splits when samples are not independent).
- Prevent information from validation or test data leaking into preprocessing or feature selection.
- Report more than one metric when one score hides important failure modes.
- Consider score variation across folds, not only the average.
- Preserve a final test set when model selection is extensive.

In [7]:
best_model = grid.best_estimator_
test_predictions = best_model.predict(X_test)
print(classification_report(y_test, test_predictions, zero_division=0))


              precision    recall  f1-score   support

           0       0.83      0.91      0.87       104
           1       0.74      0.57      0.64        46

    accuracy                           0.81       150
   macro avg       0.78      0.74      0.75       150
weighted avg       0.80      0.81      0.80       150



## Practical Workflow

1. Identify the prediction task and the consequences of different error types.
2. Inspect class balance and the structure of the observations.
3. Choose a suitable validation strategy and scoring metric.
4. Establish a baseline and examine the distribution of fold scores.
5. Tune hyperparameters with cross-validation inside the training data.
6. Compare candidate models using consistent splits and metrics.
7. Evaluate the selected model once on held-out test data.
8. Record data splits, random seeds, parameters, metrics, and software versions for reproducibility.

## Chapter 12 Summary

Cross-validation provides a more robust view of model performance than a single split. K-fold is a general-purpose method; stratification helps maintain class proportions; LOOCV makes extensive use of small datasets; and nested cross-validation separates tuning from evaluation. In scikit-learn, `cross_val_score()`, `cross_validate()`, and search estimators streamline evaluation and selection. Good generalization estimates depend on appropriate splits, meaningful metrics, and strict prevention of data leakage.